### **Cuaderno14 MCC225: Tarea individual para la Segunda Exposición Académica Programada**

#### **Evaluación experimental de sistemas multimodales con datos reales**

Este cuaderno es una base de trabajo individual. 

### **Reglas del trabajo individual**

#### **Condiciones mínimas**

El trabajo es individual. Cada estudiante debe mantener este cuaderno resuelto en su propio repositorio. No se acepta únicamente una exposición oral sin evidencia ejecutable.

El repositorio debe contener como mínimo:

```text
.
├── notebooks/
│   └── Cuaderno14_MSR_VTT33.ipynb
├── data/
│   ├── raw/
│   └── processed/
├── outputs/
│   ├── figures/
│   ├── tables/
│   └── metrics/
├── reports/
│   └── reporte_exposicion_2.md
├── requirements.txt
└── README.md
```

El estudiante debe entregar resultados reales, no capturas aisladas. El cuaderno debe producir archivos de métricas, tablas de errores, figuras y un reporte técnico breve.

### **Qué se debe demostrar**

#### Preguntas experimentales



La exposición debe sostenerse en evidencia del repositorio: métricas, tablas, ejemplos correctos, ejemplos fallidos y discusión de limitaciones.

### **Instalación sugerida**

#### **Dependencias**

Ejecute esta celda solo si el entorno no tiene las bibliotecas requeridas. En un repositorio serio, la misma lista debe aparecer en `requirements.txt`.

In [ ]:
# %pip install -q torch torchvision transformers datasets pillow pandas numpy scikit-learn matplotlib tqdm nltk evaluate accelerate

### **Configuración reproducible**

#### **Semillas, rutas y parámetros**

Esta sección fija los parámetros del experimento. Cambiar estos valores debe quedar justificado en el reporte.

In [65]:
import json
import random
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import torch


# =====================================================
# REPRODUCIBILIDAD
# =====================================================

SEED = 22514

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# =====================================================
# CONFIGURACIÓN DEL EXPERIMENTO
# =====================================================

@dataclass
class ExperimentConfig:

    ####################################################
    # DATASETS
    ####################################################

    train_metadata: str = "outputs/sports33_msr-vtt_train.csv"

    test_metadata: str = "outputs/sports33_msr-vtt_test.csv"


    ####################################################
    # MODELOS
    ####################################################

    clip_model_name: str = "openai/clip-vit-base-patch32"
    blip_model_name: str = "Salesforce/blip-image-captioning-base"
    use_blip_captioner: bool = True

    visualbert_model: str = "uclanlp/visualbert-vqa-coco-pre"

    lxmert_model: str = "unc-nlp/lxmert-base-uncased"

    ####################################################
    # FRAME DATASET
    ####################################################

    frame_context: str = "context_12"

    frame_root: str = "data/msr-vtt/frames"

    frame_metadata: str = (
        "outputs/embeddings/sports33_train_context_12_frame_metadata.csv"
    )
    ####################################################
    # TAREA
    ####################################################

    task: str = "text_to_frame_retrieval"


    ####################################################
    # RANKING TEXTO ↔ FRAME
    ####################################################

    ranking_strategy: str = "clip_similarity"

    captions_per_video: int = 20

    top_k_frames: int = 8


    ####################################################
    # FRAMES
    ####################################################

    frames_per_video: int = 12

    frame_strategy: str = "top8_clip"

    frame_strategies: tuple = (

        "central",

        "top4_clip",

        "top8_clip",

        "top16_clip",

        "average_embeddings"

    )


    ####################################################
    # PERTURBACIONES
    ####################################################

    perturbation: str = "none"

    perturbations: tuple = (

        "none",

        "reverse",

        "shuffle",

        "drop",

        "sampling"

    )


    ####################################################
    # TASA DE MUESTREO
    ####################################################

    sampling_rates: tuple = (

        1,

        2,

        4

    )


    ####################################################
    # EVALUACIÓN
    ####################################################

    batch_size: int = 16

    confidence_level: float = 0.95


    ####################################################
    # REPRODUCIBILIDAD
    ####################################################

    seed: int = SEED


    ####################################################
    # RUTAS
    ####################################################

    repo_root: str = "."


CONFIG = ExperimentConfig()


# =====================================================
# RUTAS
# =====================================================

ROOT = Path(CONFIG.repo_root).resolve()

DATA = ROOT / "data"

DATA_RAW = DATA / "raw"

DATA_VIDEOS = DATA / "videos"

DATA_FRAMES = DATA / "frames"

DATA_PROCESSED = DATA / "processed"


OUTPUTS = ROOT / "outputs"

OUTPUT_EMBEDDINGS = OUTPUTS / "embeddings"

OUTPUT_METRICS = OUTPUTS / "metrics"

OUTPUT_TABLES = OUTPUTS / "tables"

OUTPUT_FIGURES = OUTPUTS / "figures"


REPORTS = ROOT / "reports"


for folder in [

    DATA,

    DATA_RAW,

    DATA_VIDEOS,

    DATA_FRAMES,

    DATA_PROCESSED,

    OUTPUTS,

    OUTPUT_EMBEDDINGS,

    OUTPUT_METRICS,

    OUTPUT_TABLES,

    OUTPUT_FIGURES,

    REPORTS

]:

    folder.mkdir(
        parents=True,
        exist_ok=True
    )


print(f"\nDispositivo: {DEVICE}\n")

print(
    json.dumps(
        asdict(CONFIG),
        indent=2,
        ensure_ascii=False
    )
)


Dispositivo: cpu

{
  "train_metadata": "outputs/sports33_msr-vtt_train.csv",
  "test_metadata": "outputs/sports33_msr-vtt_test.csv",
  "clip_model_name": "openai/clip-vit-base-patch32",
  "blip_model_name": "Salesforce/blip-image-captioning-base",
  "use_blip_captioner": true,
  "visualbert_model": "uclanlp/visualbert-vqa-coco-pre",
  "lxmert_model": "unc-nlp/lxmert-base-uncased",
  "frame_context": "context_12",
  "frame_root": "data/msr-vtt/frames",
  "frame_metadata": "outputs/embeddings/sports33_train_context_12_frame_metadata.csv",
  "task": "text_to_frame_retrieval",
  "ranking_strategy": "clip_similarity",
  "captions_per_video": 20,
  "top_k_frames": 8,
  "frames_per_video": 12,
  "frame_strategy": "top8_clip",
  "frame_strategies": [
    "central",
    "top4_clip",
    "top8_clip",
    "top16_clip",
    "average_embeddings"
  ],
  "perturbation": "none",
  "perturbations": [
    "none",
    "reverse",
    "shuffle",
    "drop",
    "sampling"
  ],
  "sampling_rates": [
    1

In [25]:
def split_captions(captions, simbolo_separacion="|||"):
    """
    Separa captions usando un símbolo separador configurable.

    Args:
        captions (str): texto con múltiples captions unidos.
        simbolo_separacion (str): símbolo utilizado para separar captions.

    Returns:
        list: lista de captions individuales limpios.
    """

    if captions is None:
        return []

    captions_list = captions.split(simbolo_separacion)

    captions_list = [
        caption.strip()
        for caption in captions_list
        if caption.strip()
    ]

    return captions_list

In [26]:
# ============================================
# Función: construir embeddings de captions
# ============================================

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm


def build_caption_embeddings(
    sample,
    model,
    processor,
    simbolo_separacion="|||"
):
    """
    Genera embeddings de captions usando CLIP.

    Parámetros
    ----------
    sample : str
        "train" o "test"
    """

    # ----------------------------------------
    # Seleccionar metadata
    # ----------------------------------------

    if sample == "train":

        metadata_file = ROOT / CONFIG.train_metadata
        output_name = "sports33_train"

    elif sample == "test":

        metadata_file = ROOT / CONFIG.test_metadata
        output_name = "sports33_test"

    else:

        raise ValueError(
            "sample debe ser 'train' o 'test'"
        )


    df = pd.read_csv(metadata_file)


    print("\n==============================")
    print(f"Muestra: {sample}")
    print("==============================")

    print(
        "Cantidad registros:",
        len(df)
    )


    # ----------------------------------------
    # Validar captions
    # ----------------------------------------

    if "captions" not in df.columns:

        raise ValueError(
            "No existe la columna 'captions'"
        )

    
    # ----------------------------------------
    # Separar captions
    # ----------------------------------------
    
    caption_records = []
    
    for _, row in df.iterrows():
    
        captions_list = (
            str(row["captions"])
            .split(simbolo_separacion)
        )
    
        for caption_id, caption in enumerate(captions_list):
    
            caption = caption.strip()
    
            if caption:
    
                caption_records.append(
                    {
                        "caption_index": len(caption_records),
                        "video_id": row["video_id"],
                        "sport": row["sport"],
                        "caption_id": caption_id,
                        "caption": caption
                    }
                )
    
    
    caption_df = pd.DataFrame(
        caption_records
    )
    
    if caption_df.empty:
        raise ValueError(
            "No se encontraron captions después de la separación"
        )
    captions = caption_df["caption"].tolist()


    embeddings = []


    # ----------------------------------------
    # Embeddings
    # ----------------------------------------

    model.eval()

    with torch.no_grad():

        for caption in tqdm(
            captions,
            desc=f"Embeddings {sample}"
        ):

            inputs = processor(
                text=[caption],
                return_tensors="pt",
                padding=True,
                truncation=True
            )

            inputs = {
                k: v.to(DEVICE)
                for k, v in inputs.items()
            }

            text_features = model.get_text_features(
                **inputs
            )

            text_features = (
                text_features /
                text_features.norm(
                    dim=-1,
                    keepdim=True
                )
            )

            embeddings.append(
                text_features.cpu().numpy()
            )


    embeddings = np.vstack(
        embeddings
    )


    # ----------------------------------------
    # Guardar
    # ----------------------------------------

    np.save(
        OUTPUT_EMBEDDINGS /
        f"{output_name}_caption_embeddings.npy",
        embeddings
    )


    caption_df.to_csv(
        OUTPUT_EMBEDDINGS /
        f"{output_name}_caption_metadata.csv",
        index=False
    )


    print(
        "Shape:",
        embeddings.shape
    )


    return embeddings

In [6]:
# ============================================
# Copiar metadata al directorio de trabajo
# ============================================

from shutil import copy2

SOURCE_OUTPUTS = ROOT.parent / "outputs"

TRAIN_SOURCE = SOURCE_OUTPUTS / "sports33_msr-vtt_train.csv"
TEST_SOURCE = SOURCE_OUTPUTS / "sports33_msr-vtt_test.csv"

TRAIN_DEST = OUTPUTS / "sports33_msr-vtt_train.csv"
TEST_DEST = OUTPUTS / "sports33_msr-vtt_test.csv"

copy2(TRAIN_SOURCE, TRAIN_DEST)
copy2(TEST_SOURCE, TEST_DEST)

print("Archivos copiados:")
print(TRAIN_DEST)
print(TEST_DEST)

Archivos copiados:
/workspace/Exposicion3/notebook_MSR_VTT/outputs/sports33_msr-vtt_train.csv
/workspace/Exposicion3/notebook_MSR_VTT/outputs/sports33_msr-vtt_test.csv


In [16]:
# ============================================
# Copiar frames (solo si no existen)
# ============================================

from shutil import copytree

SOURCE_DATA = ROOT.parent / "data" / "msr-vtt"
DEST_DATA = DATA / "msr-vtt"

DEST_DATA.mkdir(
    parents=True,
    exist_ok=True
)

for folder in [
    "frames",
    "frames-test"
]:

    source = SOURCE_DATA / folder
    dest = DEST_DATA / folder

    if not source.exists():
        print(f"No existe el origen: {source}")
        continue

    if dest.exists():
        print(f"Ya existe: {dest}")
        continue

    copytree(
        source,
        dest
    )

    print(f"Copiado: {folder}")

print("\nProceso finalizado.")

Ya existe: /workspace/Exposicion3/notebook_MSR_VTT/data/msr-vtt/frames
Copiado: frames-test

Proceso finalizado.


In [7]:
# ============================================
# Cargar metadata train y test
# ============================================

import pandas as pd


TRAIN_CSV = ROOT / CONFIG.train_metadata

TEST_CSV = ROOT / CONFIG.test_metadata


df_train = pd.read_csv(
    TRAIN_CSV
)


df_test = pd.read_csv(
    TEST_CSV
)


print("Train:")
print(df_train.shape)

print("\nTest:")
print(df_test.shape)


print("\nColumnas:")
print(df_train.columns.tolist())

Train:
(132, 9)

Test:
(40, 9)

Columnas:
['sport', 'score', 'video_id', 'video', 'url', 'start_time', 'end_time', 'category', 'captions']


In [13]:
from transformers import CLIPModel, CLIPProcessor

model = CLIPModel.from_pretrained(
    CONFIG.clip_model_name
).to(DEVICE)

processor = CLIPProcessor.from_pretrained(
    CONFIG.clip_model_name
)

model.eval()

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

/usr/local/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPSdpaAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e

In [27]:
SEPARADOR = "|||"

caption_embeddings_train = build_caption_embeddings(
    "train",
    model,
    processor,
    simbolo_separacion=SEPARADOR
)


caption_embeddings_test = build_caption_embeddings(
    "test",
    model,
    processor,
    simbolo_separacion=SEPARADOR
)


Muestra: train
Cantidad registros: 132


Embeddings train: 100%|█████████████████████| 2640/2640 [01:10<00:00, 37.60it/s]


Shape: (2640, 512)

Muestra: test
Cantidad registros: 40


Embeddings test: 100%|████████████████████████| 800/800 [00:25<00:00, 31.14it/s]


Shape: (800, 512)


Función para obtener embeddings de los frames 

In [17]:
# ============================================
# Función: construir embeddings de frames
# ============================================

from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm


def build_frame_embeddings(
    sample,
    context,
    model,
    processor
):
    """
    Genera embeddings CLIP para todos los frames.

    Parámetros
    ----------
    sample : str
        "train" o "test"

    context : str
        Ejemplo:
            "context_12"
            "context_8"
            "context_4"
    """

    # ============================================
    # Seleccionar carpeta de frames
    # ============================================

    if sample == "train":

        frame_root = (
            DATA /
            "msr-vtt" /
            "frames" /
            context
        )

        output_prefix = f"sports33_train_{context}"

    elif sample == "test":

        frame_root = (
            DATA /
            "msr-vtt" /
            "frames-test" /
            context
        )

        output_prefix = f"sports33_test_{context}"

    else:

        raise ValueError(
            "sample debe ser 'train' o 'test'"
        )


    if not frame_root.exists():

        raise FileNotFoundError(
            frame_root
        )


    print("\n==============================")
    print(f"Muestra : {sample}")
    print(f"Contexto: {context}")
    print("==============================\n")


    embeddings = []

    metadata = []


    sports = sorted(

        folder

        for folder in frame_root.iterdir()

        if folder.is_dir()

    )


    model.eval()


    with torch.no_grad():

        for sport_dir in sports:

            images = sorted(

                sport_dir.glob("*.jpg")

            )

            print(
                f"{sport_dir.name}: {len(images)} frames"
            )


            for image_path in tqdm(
                images,
                leave=False
            ):

                image = Image.open(
                    image_path
                ).convert("RGB")


                inputs = processor(

                    images=image,

                    return_tensors="pt"

                )


                inputs = {

                    k: v.to(DEVICE)

                    for k, v in inputs.items()

                }


                image_features = model.get_image_features(
                    **inputs
                )


                image_features = (

                    image_features /

                    image_features.norm(
                        dim=-1,
                        keepdim=True
                    )

                )


                embeddings.append(
                    image_features.cpu().numpy()
                )


                video_id, frame_number = (
                    image_path.stem.rsplit(
                        "_frame_",
                        1
                    )
                )


                metadata.append({

                    "video_id": video_id,

                    "sport": sport_dir.name,

                    "frame_file": image_path.name,

                    "frame_number": int(
                        frame_number
                    )

                })


    embeddings = np.vstack(
        embeddings
    )


    metadata = pd.DataFrame(
        metadata
    )


    embedding_file = (

        OUTPUT_EMBEDDINGS /

        f"{output_prefix}_frame_embeddings.npy"

    )


    metadata_file = (

        OUTPUT_EMBEDDINGS /

        f"{output_prefix}_frame_metadata.csv"

    )


    np.save(

        embedding_file,

        embeddings

    )


    metadata.to_csv(

        metadata_file,

        index=False

    )


    print("\nEmbeddings:", embeddings.shape)

    print(
        "Guardado:",
        embedding_file
    )

    print(
        "Metadata:",
        metadata_file
    )


    return embeddings, metadata

In [18]:
# ============================================
# LLamadas para Embeddings train
# ============================================

frame_embeddings_train, frame_metadata_train = build_frame_embeddings(
    sample="train",
    context="context_12",
    model=model,
    processor=processor
)


Muestra : train
Contexto: context_12

basketball: 396 frames


soccer: 396 frames


swimming: 384 frames


tennis: 396 frames



Embeddings: (1572, 512)
Guardado: /workspace/Exposicion3/notebook_MSR_VTT/outputs/embeddings/sports33_train_context_12_frame_embeddings.npy
Metadata: /workspace/Exposicion3/notebook_MSR_VTT/outputs/embeddings/sports33_train_context_12_frame_metadata.csv


In [19]:
# ============================================
# Embeddings test
# ============================================

frame_embeddings_test, frame_metadata_test = build_frame_embeddings(
    sample="test",
    context="context_12",
    model=model,
    processor=processor
)


Muestra : test
Contexto: context_12

basketball: 60 frames


soccer: 60 frames


swimming: 36 frames


tennis: 60 frames



Embeddings: (216, 512)
Guardado: /workspace/Exposicion3/notebook_MSR_VTT/outputs/embeddings/sports33_test_context_12_frame_embeddings.npy
Metadata: /workspace/Exposicion3/notebook_MSR_VTT/outputs/embeddings/sports33_test_context_12_frame_metadata.csv


In [29]:
# ============================================
# Función: ranking CLIP por video
# ============================================

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F


def build_clip_frame_ranking(
    sample,
    context,
    top_k=None
):
    """
    Ranking CLIP entre caption y frames
    del mismo video.

    Parameters
    ----------
    sample : str
        "train" o "test"

    context : str
        Ejemplo:
        context_12

    top_k : int
        Número de frames a recuperar.
    """

    if top_k is None:
        top_k = CONFIG.top_k_frames


    # ============================================
    # Cargar embeddings
    # ============================================

    if sample == "train":

        caption_embeddings = np.load(
            OUTPUT_EMBEDDINGS /
            "sports33_train_caption_embeddings.npy"
        )

        frame_embeddings = np.load(
            OUTPUT_EMBEDDINGS /
            f"sports33_train_{context}_frame_embeddings.npy"
        )

        caption_metadata = pd.read_csv(
            OUTPUT_EMBEDDINGS /
            "sports33_train_caption_metadata.csv"
        )

        frame_metadata = pd.read_csv(
            OUTPUT_EMBEDDINGS /
            f"sports33_train_{context}_frame_metadata.csv"
        )

        output_file = (
            OUTPUT_TABLES /
            f"sports33_train_{context}_clip_ranking.csv"
        )

    elif sample == "test":

        caption_embeddings = np.load(
            OUTPUT_EMBEDDINGS /
            "sports33_test_caption_embeddings.npy"
        )

        frame_embeddings = np.load(
            OUTPUT_EMBEDDINGS /
            f"sports33_test_{context}_frame_embeddings.npy"
        )

        caption_metadata = pd.read_csv(
            OUTPUT_EMBEDDINGS /
            "sports33_test_caption_metadata.csv"
        )

        frame_metadata = pd.read_csv(
            OUTPUT_EMBEDDINGS /
            f"sports33_test_{context}_frame_metadata.csv"
        )

        output_file = (
            OUTPUT_TABLES /
            f"sports33_test_{context}_clip_ranking.csv"
        )

    else:

        raise ValueError(
            "sample debe ser train o test"
        )


    # ============================================
    # Tensor
    # ============================================

    caption_embeddings = torch.tensor(
        caption_embeddings,
        dtype=torch.float32
    )

    frame_embeddings = torch.tensor(
        frame_embeddings,
        dtype=torch.float32
    )


    ranking_rows = []


    # ============================================
    # Ranking por video
    # ============================================

    for caption_idx, row in caption_metadata.iterrows():

        video_id = row["video_id"]

        caption = row["caption"]
     
        caption_id = row["caption_id"]
        # Frames del mismo video
        frames_video = frame_metadata[
            frame_metadata.video_id == video_id
        ]


        if len(frames_video) == 0:

            print(
                f"No hay frames para {video_id}"
            )

            continue


        frame_indices = frames_video.index.tolist()


        frame_vectors = frame_embeddings[
            frame_indices
        ]


        caption_vector = caption_embeddings[
            caption_idx
        ].unsqueeze(0)


        similarity = F.cosine_similarity(

            caption_vector,

            frame_vectors,

            dim=1

        )


        values, indices = torch.topk(

            similarity,

            k=min(
                top_k,
                len(frames_video)
            )

        )


        for rank, (
            score,
            local_idx
        ) in enumerate(

            zip(
                values.tolist(),
                indices.tolist()
            ),

            start=1

        ):

            frame = frames_video.iloc[
                local_idx
            ]


            ranking_rows.append({

                "video_id": video_id,

                "sport": frame["sport"],

                "caption_id": caption_id,

                "caption": caption,

                "rank": rank,

                "similarity": score,

                "frame_file": frame["frame_file"],

                "frame_number": frame["frame_number"]

            })


    ranking = pd.DataFrame(
        ranking_rows
    )

    # ============================================
    # Mejor caption encontrado por video
    # ============================================
    
    best_frame_caption_match = (
        ranking
        .sort_values(
            "similarity",
            ascending=False
        )
        .groupby(
            "video_id"
        )
        .first()
        .reset_index()
    )
    ranking.to_csv(

        output_file,

        index=False

    )
    best_frame_caption_match.to_csv(
        OUTPUT_TABLES /
        f"{sample}_{context}_best_frame_caption_match.csv",
        index=False
    )

    print(
        f"\nRanking guardado:\n{output_file}"
    )

    print(
        f"Filas: {len(ranking)}"
    )
    print(
        "\nMejor similitud promedio por frame-caption:",
        best_frame_caption_match["similarity"].mean()
    )

    return ranking

In [31]:
#llamadas RANKING
ranking_train = build_clip_frame_ranking(
    sample="train",
    context="context_12"
)
ranking_test = build_clip_frame_ranking(
    sample="test",
    context="context_12"
)

No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759
No hay frames para video4759

Ranking guardado:
/workspace/Exposicion3/notebook_MSR_VTT/outputs/tables/sports33_train_context_12_clip_ranking.csv
Filas: 20960

Mejor similitud promedio por frame-caption: 0.35036182767562285
No hay frames para video1084
No hay frames para video1084
No hay frames para video1084
No hay frames para video1084
No hay frames para video1084
No hay frames para video1084
No hay frames para video1084
No hay frames para vid

In [39]:
# ============================================
# Generador BLIP captions por frame
# ============================================

from typing import Sequence, List

import pandas as pd
import torch

from PIL import Image
from tqdm import tqdm


def generate_blip_captions(
    image_paths: Sequence[str],
    batch_size: int
) -> List[str]:
    """
    Genera captions usando BLIP para una lista de frames.
    """

    from transformers import (
        BlipProcessor,
        BlipForConditionalGeneration
    )


    processor = BlipProcessor.from_pretrained(
        CONFIG.blip_model_name
    )


    model = BlipForConditionalGeneration.from_pretrained(
        CONFIG.blip_model_name
    ).to(DEVICE)


    model.eval()


    captions = []


    for start in tqdm(
        range(
            0,
            len(image_paths),
            batch_size
        ),
        desc="Generando captions BLIP"
    ):


        batch_paths = image_paths[
            start:start + batch_size
        ]


        images = [
            Image.open(path)
            .convert("RGB")
            for path in batch_paths
        ]


        inputs = processor(
            images=images,
            return_tensors="pt",
            padding=True
        )


        inputs = {
            k: v.to(DEVICE)
            for k, v in inputs.items()
        }


        with torch.no_grad():

            generated = model.generate(
                **inputs,
                max_new_tokens=32
            )


        decoded = processor.batch_decode(
            generated,
            skip_special_tokens=True
        )


        captions.extend(
            [
                x.strip()
                for x in decoded
            ]
        )


    return captions

In [41]:
train_frame_metadata = pd.read_csv(
    OUTPUT_EMBEDDINGS /
    "sports33_train_context_12_frame_metadata.csv"
)

In [46]:
# ============================================
# Construir CSV captions BLIP por frame
# ============================================

def build_blip_caption_csv(
    frame_metadata,
    output_file,
    batch_size,
    exclude_sport="golf"
):
    """
    Genera captions BLIP por frame.

    Entrada:
        video_id
        sport
        frame_file
        frame_number

    Salida:
        video_id
        frame_file
        sport
        caption_by_blip
    """


    df = frame_metadata.copy()


    # ----------------------------------------
    # Eliminar golf
    # ----------------------------------------

    df = df[
        df["sport"]
        .str.lower()
        != exclude_sport.lower()
    ].copy()


    print(
        "Frames después de eliminar golf:",
        len(df)
    )


    print(
        df["sport"].value_counts()
    )

    # ----------------------------------------
    # Construir ruta de cada frame
    # ----------------------------------------
    
    frame_root = (
        ROOT /
        "data" /
        "msr-vtt" /
        "frames" /
        "context_12"
    )
    
    df["image_path"] = df.apply(
        lambda row: str(
            frame_root /
            row["sport"] /
            row["frame_file"]
        ),
        axis=1
    )

    # Verificación primera imagen

    print(
        "\nPrimera imagen:"
    )

    print(
        df["image_path"].iloc[0]
    )


    # ----------------------------------------
    # Generar captions BLIP
    # ----------------------------------------

    captions = generate_blip_captions(
        df["image_path"].tolist(),
        batch_size
    )


    df["caption_by_blip"] = captions


    # ----------------------------------------
    # Guardar CSV
    # ----------------------------------------

    blip_df = df[
        [
            "video_id",
            "frame_file",
            "sport",
            "frame_number",
            "caption_by_blip"
        ]
    ]


    blip_df.to_csv(
        output_file,
        index=False
    )


    print(
        "\nCSV BLIP guardado:"
    )

    print(
        output_file
    )


    return blip_df

In [47]:
train_frame_metadata = pd.read_csv(
    OUTPUT_EMBEDDINGS /
    "sports33_train_context_12_frame_metadata.csv"
)


train_blip_df = build_blip_caption_csv(
    frame_metadata=train_frame_metadata,

    output_file=
        OUTPUT_TABLES /
        "sports33_train_context_12_blip_captions.csv",

    batch_size=CONFIG.batch_size,

    exclude_sport="golf"
)

Frames después de eliminar golf: 1572
sport
basketball    396
soccer        396
tennis        396
swimming      384
Name: count, dtype: int64

Primera imagen:
/workspace/Exposicion3/notebook_MSR_VTT/data/msr-vtt/frames/context_12/basketball/video1623_frame_000.jpg


/usr/local/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Generando captions BLIP: 100%|████████████████| 99/99 [2:02:09<00:00, 74.04s/it]



CSV BLIP guardado:
/workspace/Exposicion3/notebook_MSR_VTT/outputs/tables/sports33_train_context_12_blip_captions.csv


In [48]:
# =====================================================
# CNN Visual Encoder (ResNet50)
# =====================================================

import torch
import torch.nn as nn

from torchvision import models
from torchvision import transforms

from PIL import Image


# ---------------------------------------------
# Modelo
# ---------------------------------------------

resnet = models.resnet50(
    weights=models.ResNet50_Weights.DEFAULT
)

# eliminar la capa de clasificación
cnn_encoder = nn.Sequential(
    *list(resnet.children())[:-1]
)

cnn_encoder = cnn_encoder.to(DEVICE)
cnn_encoder.eval()


# ---------------------------------------------
# Transformaciones
# ---------------------------------------------

transform = transforms.Compose([

    transforms.Resize(256),

    transforms.CenterCrop(224),

    transforms.ToTensor(),

    transforms.Normalize(

        mean=[0.485,0.456,0.406],

        std=[0.229,0.224,0.225]

    )

])

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████████████████████████████████| 97.8M/97.8M [00:05<00:00, 19.4MB/s]


Construcción de 12 tokens visuales
03_visual_tokens.py

Este script será ejecutado una sola vez.

Su objetivo es generar los embeddings CNN para cada uno de los 12 frames.

Video760

↓

Frame000

↓

CNN

↓

2048

Frame001

↓

CNN

↓

2048

...

↓

Frame011

↓

CNN

↓

2048

↓

Tensor (12,2048)

↓

Guardar

In [50]:
import numpy as np
import pandas as pd

from PIL import Image

import torch
from tqdm import tqdm
# =====================================================
# Construir tokens visuales
# =====================================================
def build_visual_tokens(
    sample,
    context,
    cnn_encoder,
    transform
):

    if sample == "train":

        metadata = pd.read_csv(

            OUTPUT_EMBEDDINGS /
            f"sports33_train_{context}_frame_metadata.csv"

        )

        output_name = (
            OUTPUT_EMBEDDINGS /
            "sports33_train_visual_tokens.npy"
        )

    elif sample == "test":

        metadata = pd.read_csv(

            OUTPUT_EMBEDDINGS /
            f"sports33_test_{context}_frame_metadata.csv"

        )

        output_name = (
            OUTPUT_EMBEDDINGS /
            "sports33_test_visual_tokens.npy"
        )

    else:

        raise ValueError()

pipeline: extracción de representaciones visuales por frame usando ResNet50.
Frames
(context_12)
      |
      v
ResNet50
      |
      v
visual_tokens_context_12.npy
      |
      v
+ Text embeddings / captions
      |
      v
Dataset multimodal
      |
      v
VisualBERT / LXMERT
      |
      v
Ranking texto ↔ frames/videos
      |
      v
Evaluación + perturbaciones

El siguiente paso lógico es construir el dataset multimodal.
Paso 1  generar embeddings de texto, enfocandonos en convertir tus captions a vectores.
Para VisualBERT/LXMERT necesitamos:

tokens de texto (input_ids)
máscara de atención (attention_mask)

usando el tokenizer de BERT.

In [53]:
# =====================================================
# Construcción de visual tokens por video
# =====================================================

import pandas as pd
import numpy as np
from tqdm import tqdm
from PIL import Image


# Cargar metadata de frames
metadata = pd.read_csv(
    ROOT / CONFIG.frame_metadata
)


visual_tokens = {}


video_ids = sorted(
    metadata["video_id"].unique()
)


for video_id in tqdm(
    video_ids,
    desc="Construyendo visual tokens"
):

    # ---------------------------------------------
    # Frames del video
    # ---------------------------------------------

    frames = (
        metadata[
            metadata["video_id"] == video_id
        ]
        .sort_values("frame_number")
    )


    if len(frames) != CONFIG.frames_per_video:

        print(
            f"Advertencia: {video_id} tiene "
            f"{len(frames)} frames."
        )


    embeddings = []


    # ---------------------------------------------
    # Embedding CNN de cada frame
    # ---------------------------------------------

    for _, row in frames.iterrows():

        image_path = (
            ROOT /
            CONFIG.frame_root /
            CONFIG.frame_context /
            row["sport"] /
            row["frame_file"]
        )


        image = (
            Image.open(image_path)
            .convert("RGB")
        )


        image = transform(image)

        image = (
            image
            .unsqueeze(0)
            .to(DEVICE)
        )


        with torch.no_grad():

            feature = cnn_encoder(image)


        feature = feature.squeeze()


        embeddings.append(
            feature.cpu().numpy()
        )


    # ---------------------------------------------
    # Guardar tokens del video
    # ---------------------------------------------

    visual_tokens[video_id] = np.stack(
        embeddings
    )



# =====================================================
# Guardar visual tokens
# =====================================================

output_name = (
    OUTPUT_EMBEDDINGS /
    f"visual_tokens_{CONFIG.frame_context}.npy"
)


np.save(
    output_name,
    visual_tokens,
    allow_pickle=True
)


print("\nVisual tokens guardados en:")
print(output_name)


print(
    "\nCantidad de videos:",
    len(visual_tokens)
)


primer_video = next(iter(visual_tokens))


print(
    "\nEjemplo:",
    primer_video
)


print(
    "Shape:",
    visual_tokens[primer_video].shape
)

Construyendo visual tokens:   0%|                       | 0/131 [00:00<?, ?it/s]

Advertencia: video1103 tiene 12 frames.


Construyendo visual tokens:   1%|               | 1/131 [00:02<04:46,  2.20s/it]

Advertencia: video1241 tiene 12 frames.


Construyendo visual tokens:   2%|▏              | 2/131 [00:03<04:08,  1.93s/it]

...

Advertencia: video9363 tiene 12 frames.


Construyendo visual tokens:  99%|████████████▉| 130/131 [04:16<00:02,  2.36s/it]

Advertencia: video9650 tiene 12 frames.


Construyendo visual tokens: 100%|█████████████| 131/131 [04:18<00:00,  1.97s/it]



Visual tokens guardados en:
/workspace/Exposicion3/notebook_MSR_VTT/outputs/embeddings/visual_tokens_context_12.npy

Cantidad de videos: 131

Ejemplo: video1103
Shape: (12, 2048)


In [56]:
# =====================================================
# Verificar shape de visual tokens
# =====================================================

print("Cantidad de videos:")
print(len(visual_tokens))


# tomar un video cualquiera
video_id = next(iter(visual_tokens))

print("\nVideo ejemplo:")
print(video_id)


print("\nShape de visual tokens:")
print(visual_tokens[video_id].shape)
metadata["video_id"].nunique()


train = pd.read_csv(
    ROOT / CONFIG.train_metadata
)

print(train.columns)
print(train.head(2))

Cantidad de videos:
131

Video ejemplo:
video1103

Shape de visual tokens:
(12, 2048)
Index(['sport', 'score', 'video_id', 'video', 'url', 'start_time', 'end_time',
       'category', 'captions'],
      dtype='object')
        sport  score   video_id          video  \
0  basketball     16  video1623  video1623.mp4   
1  basketball     16  video1667  video1667.mp4   

                                           url  start_time  end_time  \
0  https://www.youtube.com/watch?v=1IIDthMD09M      107.89    119.00   
1  https://www.youtube.com/watch?v=cpCrFn7MQ8k      200.45    213.49   

   category                                           captions  
0         3  a basketball game being played ||| a basketbal...  
1         1  a basketball player does a slam dunk ||| a man...  


In [58]:
# =====================================================
# 04_build_multimodal_dataset.py
# Parte 1
# Cargar datos
# =====================================================

import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from tqdm import tqdm

from transformers import AutoTokenizer


# =====================================================
# CONFIGURACIÓN
# =====================================================

print("=" * 60)
print("Construcción del Dataset Multimodal")
print("=" * 60)


# -----------------------------------------------------
# Tokenizer
# -----------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)


# -----------------------------------------------------
# Rutas
# -----------------------------------------------------

visual_tokens_file = (
    OUTPUT_EMBEDDINGS /
    f"visual_tokens_{CONFIG.frame_context}.npy"
)

ranking_file = (
    OUTPUT_TABLES /
    f"sports33_train_{CONFIG.frame_context}_clip_ranking.csv"
)

blip_file = (
    OUTPUT_TABLES /
    f"sports33_train_{CONFIG.frame_context}_blip_captions.csv"
)

output_dataset = (
    DATA_PROCESSED /
    f"multimodal_train_{CONFIG.frame_context}.pt"
)


print("\nArchivos")

print(visual_tokens_file)
print(ranking_file)
print(blip_file)


# =====================================================
# Cargar visual tokens
# =====================================================

print("\nCargando visual tokens...")

visual_tokens = np.load(
    visual_tokens_file,
    allow_pickle=True
).item()

print(
    "Videos:",
    len(visual_tokens)
)


# =====================================================
# Cargar ranking CLIP
# =====================================================

print("\nCargando ranking...")

ranking = pd.read_csv(
    ranking_file
)

print(
    "Filas ranking:",
    len(ranking)
)

print(
    ranking.head()
)


# =====================================================
# Cargar captions BLIP
# =====================================================

print("\nCargando captions BLIP...")

blip = pd.read_csv(
    blip_file
)

print(
    "Filas BLIP:",
    len(blip)
)

print(
    blip.head()
)


# =====================================================
# Verificar columnas
# =====================================================

print("\nColumnas Ranking")

print(
    ranking.columns.tolist()
)

print("\nColumnas BLIP")

print(
    blip.columns.tolist()
)

Construcción del Dataset Multimodal

Archivos
/workspace/Exposicion3/notebook_MSR_VTT/outputs/embeddings/visual_tokens_context_12.npy
/workspace/Exposicion3/notebook_MSR_VTT/outputs/tables/sports33_train_context_12_clip_ranking.csv
/workspace/Exposicion3/notebook_MSR_VTT/outputs/tables/sports33_train_context_12_blip_captions.csv

Cargando visual tokens...
Videos: 131

Cargando ranking...
Filas ranking: 20960
    video_id       sport  caption_id                         caption  rank  \
0  video1623  basketball           0  a basketball game being played     1   
1  video1623  basketball           0  a basketball game being played     2   
2  video1623  basketball           0  a basketball game being played     3   
3  video1623  basketball           0  a basketball game being played     4   
4  video1623  basketball           0  a basketball game being played     5   

   similarity               frame_file  frame_number  
0    0.271185  video1623_frame_003.jpg             3  
1    0.26

In [59]:
# =====================================================
# Parte 2
# Merge entre Ranking + BLIP + Visual Tokens
# =====================================================

print("\n" + "=" * 60)
print("Uniendo información")
print("=" * 60)


# -----------------------------------------------------
# Merge Ranking + BLIP
# -----------------------------------------------------

multimodal = ranking.merge(

    blip,

    on=[
        "video_id",
        "frame_file",
        "sport",
        "frame_number"
    ],

    how="left"

)


print("\nFilas después del merge:")

print(len(multimodal))


print("\nColumnas:")

print(multimodal.columns.tolist())


# -----------------------------------------------------
# Verificar captions BLIP faltantes
# -----------------------------------------------------

missing = (
    multimodal["caption_by_blip"]
    .isna()
    .sum()
)

print("\nCaptions BLIP faltantes:")

print(missing)


# -----------------------------------------------------
# Verificar visual tokens disponibles
# -----------------------------------------------------

video_ids_tokens = set(
    visual_tokens.keys()
)

video_ids_dataset = set(
    multimodal["video_id"]
)

faltantes = sorted(

    video_ids_dataset -
    video_ids_tokens

)

print("\nVideos sin visual tokens:")

print(len(faltantes))


if len(faltantes) > 0:

    print(faltantes[:20])


# -----------------------------------------------------
# Conservar únicamente videos válidos
# -----------------------------------------------------

multimodal = multimodal[

    multimodal["video_id"].isin(
        video_ids_tokens
    )

].copy()


print("\nFilas finales:")

print(len(multimodal))


print(
    "\nVideos únicos:",
    multimodal["video_id"].nunique()
)


# -----------------------------------------------------
# Ordenar
# -----------------------------------------------------

multimodal = multimodal.sort_values(

    [

        "video_id",

        "rank",

        "frame_number"

    ]

).reset_index(drop=True)


print("\nPrimeras filas")

display(

    multimodal.head()

)


Uniendo información

Filas después del merge:
20960

Columnas:
['video_id', 'sport', 'caption_id', 'caption', 'rank', 'similarity', 'frame_file', 'frame_number', 'caption_by_blip']

Captions BLIP faltantes:
0

Videos sin visual tokens:
0

Filas finales:
20960

Videos únicos: 131

Primeras filas


,video_id,sport,caption_id,caption,rank,similarity,frame_file,frame_number,caption_by_blip
0,video1103,soccer,0,a man cooking food,1,0.185026,video1103_frame_000.jpg,0,a soccer game is being played on the field
1,video1103,soccer,1,a man is playing soccer,1,0.277449,video1103_frame_000.jpg,0,a soccer game is being played on the field
2,video1103,soccer,2,a man missing a goal in soccer with his team m...,1,0.300338,video1103_frame_000.jpg,0,a soccer game is being played on the field
3,video1103,soccer,5,a soccer player kicks a goal,1,0.302797,video1103_frame_000.jpg,0,a soccer game is being played on the field
4,video1103,soccer,9,compilation of popular soccer clips,1,0.313305,video1103_frame_000.jpg,0,a soccer game is being played on the field


In [60]:
# =====================================================
# Parte 3
# Construcción del dataset multimodal
# =====================================================

print("\n" + "=" * 60)
print("Construyendo Dataset Multimodal")
print("=" * 60)


# -----------------------------------------------------
# Elegir fuente de texto
# -----------------------------------------------------

# "caption"           -> MSR-VTT
# "caption_by_blip"   -> BLIP

TEXT_SOURCE = "caption_by_blip"


print("\nFuente de texto:")

print(TEXT_SOURCE)


# -----------------------------------------------------
# Longitud máxima
# -----------------------------------------------------

MAX_LENGTH = 40


dataset = []


# =====================================================
# Construcción
# =====================================================

for _, row in tqdm(

    multimodal.iterrows(),

    total=len(multimodal),

    desc="Creando dataset"

):

    video_id = row["video_id"]


    # ---------------------------------------------
    # Texto
    # ---------------------------------------------

    text = row[TEXT_SOURCE]

    if pd.isna(text):

        continue

    text = str(text)


    encoded = tokenizer(

        text,

        padding="max_length",

        truncation=True,

        max_length=MAX_LENGTH,

        return_tensors="pt"

    )


    # ---------------------------------------------
    # Visual Tokens
    # ---------------------------------------------

    visual = visual_tokens[video_id]

    visual = torch.tensor(

        visual,

        dtype=torch.float32

    )


    # ---------------------------------------------
    # Registro
    # ---------------------------------------------

    sample = {

        "video_id":

            video_id,

        "sport":

            row["sport"],

        "caption":

            row["caption"],

        "caption_by_blip":

            row["caption_by_blip"],

        "frame_file":

            row["frame_file"],

        "frame_number":

            int(row["frame_number"]),

        "rank":

            int(row["rank"]),

        "similarity":

            float(row["similarity"]),


        "visual_embeds":

            visual,


        "input_ids":

            encoded["input_ids"].squeeze(0),

        "attention_mask":

            encoded["attention_mask"].squeeze(0)

    }


    dataset.append(

        sample

    )


print("\nDataset construido")

print(

    len(dataset)

)


Construyendo Dataset Multimodal

Fuente de texto:
caption_by_blip


Creando dataset: 100%|███████████████████| 20960/20960 [00:41<00:00, 504.69it/s]


Dataset construido
20960


In [61]:
print(dataset[0].keys())

dict_keys(['video_id', 'sport', 'caption', 'caption_by_blip', 'frame_file', 'frame_number', 'rank', 'similarity', 'visual_embeds', 'input_ids', 'attention_mask'])


In [63]:
sample = dataset[0]

print("Video:", sample["video_id"])
print("Deporte:", sample["sport"])

print("\nVisual Embeds:")
print(sample["visual_embeds"].shape)

print("\nInput IDs:")
print(sample["input_ids"].shape)

print("\nAttention Mask:")
print(sample["attention_mask"].shape)

print("\nCaption original:")
print(sample["caption"])

print("\nCaption BLIP:")
print(sample["caption_by_blip"])

Video: video1103
Deporte: soccer

Visual Embeds:
torch.Size([12, 2048])

Input IDs:
torch.Size([40])

Attention Mask:
torch.Size([40])

Caption original:
a man cooking food

Caption BLIP:
a soccer game is being played on the field


In [64]:
# =====================================================
# Parte 4
# Guardar Dataset Multimodal
# =====================================================

print("\n" + "=" * 60)
print("Guardando Dataset")
print("=" * 60)


# -----------------------------------------------------
# Guardar
# -----------------------------------------------------

torch.save(
    dataset,
    output_dataset
)

print("\nDataset guardado en:")

print(output_dataset)


# =====================================================
# Estadísticas
# =====================================================

print("\n" + "=" * 60)
print("Resumen")
print("=" * 60)

print(
    "Número de muestras:",
    len(dataset)
)

print(
    "Número de videos:",
    len(
        set(
            d["video_id"]
            for d in dataset
        )
    )
)


sports = {}

for sample in dataset:

    sport = sample["sport"]

    sports[sport] = sports.get(
        sport,
        0
    ) + 1


print("\nDistribución")

for sport in sorted(sports):

    print(
        f"{sport:12s}: {sports[sport]}"
    )


# =====================================================
# Validación
# =====================================================

print("\n" + "=" * 60)
print("Validación")
print("=" * 60)

sample = random.choice(dataset)

print("\nVideo:")

print(sample["video_id"])

print("\nFrame:")

print(sample["frame_file"])

print("\nCaption:")

print(sample["caption"])

print("\nCaption BLIP:")

print(sample["caption_by_blip"])

print("\nVisual Embeds:")

print(sample["visual_embeds"].shape)

print("\nInput IDs:")

print(sample["input_ids"].shape)

print("\nAttention Mask:")

print(sample["attention_mask"].shape)

print("\nSimilarity:")

print(sample["similarity"])

print("\nRank:")

print(sample["rank"])


print("\nDataset listo para VisualBERT / LXMERT")


Guardando Dataset

Dataset guardado en:
/workspace/Exposicion3/notebook_MSR_VTT/data/processed/multimodal_train_context_12.pt

Resumen
Número de muestras: 20960
Número de videos: 131

Distribución
basketball  : 5280
soccer      : 5280
swimming    : 5120
tennis      : 5280

Validación

Video:
video4337

Frame:
video4337_frame_000.jpg

Caption:
teams are playing basketball on the court

Caption BLIP:
a basketball game with a player in the middle of the court

Visual Embeds:
torch.Size([12, 2048])

Input IDs:
torch.Size([40])

Attention Mask:
torch.Size([40])

Similarity:
0.2681252360343933

Rank:
2

Dataset listo para VisualBERT / LXMERT


### **Congelar solo entrenar la capa final**
Entrenar solo la capa final (con VisualBERT y LXMERT congelados) tiene varias ventajas para tu caso:

 Es totalmente válido metodológicamente.
 Evita sobreajuste, ya que tu dataset es relativamente pequeño.
 Se puede ejecutar en CPU.
 La comparación entre ambos modelos es justa.
#### **Revisión cualitativa inicial**


